# Finding the best model — corrected evaluation

این نسخه، روش **Word2Vec + چهار خانوادهٔ مدل نسخهٔ اصلی** را حفظ می‌کند.
ترتیب کار: بررسی کیفیت متن خام ← تقسیم طبقه‌بندی‌شده ← پاکسازی مستقل هر بخش
← آموزش Word2Vec و مدل فقط با آموزش ← انتخاب پارامتر، آستانه و مدل بدون تست
← ارزیابی نهایی مدل منتخب روی تست.

- سهم اولیه: **۶۴٪ آموزش، ۱۶٪ اعتبارسنجی، ۲۰٪ تست**؛ حذف متن‌های تهی یا
  تکراری پس از پاکسازی ممکن است این نسبت‌ها را کمی تغییر دهد.
- در `RandomizedSearchCV` کل پایپ‌لاین برازش می‌شود؛ Word2Vec در هر فولد
  فقط متن‌های بخش آموزشی آن فولد را می‌بیند.
- آستانه و مدل با **F1 کلاس ۱** در اعتبارسنجی انتخاب می‌شوند. معیار جست‌وجوی
  پارامترها نیز F1 کلاس ۱ است؛ برای تغییر آن، قبل از اجرا تنظیمات را عوض کنید.
- برچسب ثابت: `0 = non-suicide` و `1 = suicide`.

**اجرا:** از `Restart Kernel → Run All` استفاده کنید. مسیر CSV را در `DATA_PATH`
تنظیم کنید، یا فایل را کنار نوت‌بوک/در ورودی Kaggle قرار دهید. در نبود فایل محلی،
دانلود با `kagglehub` انجام می‌شود و اینترنت لازم است. جست‌وجوی کامل می‌تواند
زمان‌بر باشد؛ `SEARCH_ITERATIONS` و تنظیمات Word2Vec قابل تغییرند.

خروجی‌های اجرای قدیمی حذف شده‌اند؛ اعداد قبلی متعلق به روش دارای نشت بودند.
دیتاست CSV همراه فایل ارسالی نبود، بنابراین این نسخه روی دیتاست اصلی دوباره
آموزش داده نشده است. آزمون اجرایی جداگانه با دادهٔ مصنوعی انجام شده است؛
هیچ معیار مصنوعی به‌عنوان نتیجهٔ پژوهش در این فایل ذخیره نشده است.

## ۱. وابستگی‌ها و تنظیمات

کد به NLTK، فایل دانلودی stopwords، python-bidi یا emoji نیاز ندارد.
پاکسازی از فهرست توقف استفاده نمی‌کند تا نفی‌هایی مانند `not`, `no`, `never`
حذف نشوند. کلمات هشتگ و اعداد حفظ می‌شوند؛ URL، HTML، ایمیل، نام کاربری و
نشانه‌های غیرواژگانی حذف می‌شوند. قواعد فارسی غیرضروری و اصلاح فاصلهٔ
دارای خطای تکرار توکن از نسخهٔ اصلی حذف شده‌اند.

In [ ]:
# If needed, uncomment in a notebook cell, install, then restart the kernel:
# %pip install numpy pandas scipy scikit-learn matplotlib seaborn gensim xgboost kagglehub
# Optional visualization only: %pip install wordcloud


In [ ]:
from pathlib import Path
from collections import Counter
from hashlib import md5
import html
import importlib.metadata
import os
import re
import tempfile
import time
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from gensim.models import Word2Vec
from joblib import Memory
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.utils.validation import check_is_fitted

SEED = 42
DATA_PATH = os.environ.get('SUICIDE_DATA_PATH') or None
# Example: DATA_PATH = r'C:\data\Suicide_Detection.csv'
TEST_SIZE = 0.20
VALIDATION_FRACTION_OF_DEVELOPMENT = 0.20  # 0.8 * 0.2 = 16% overall
CV_FOLDS = 3
SEARCH_ITERATIONS = 10
SEARCH_SCORING = 'f1'  # positive class = 1, evaluated at the estimator's default threshold
RUN_XGBOOST = True
SEARCH_N_JOBS = 1  # portable notebook classes; avoids nested parallelism
MODEL_N_JOBS = 2
USE_PIPELINE_CACHE = True  # reuses identical fold-specific transforms, never other folds
W2V_CONFIG = dict(vector_size=100, window=5, min_count=2, sg=1,
                  epochs=5, workers=1, seed=SEED)
THRESHOLDS = np.linspace(0.0, 1.0, 101)
LABEL_MAP = {'non-suicide': 0, 'suicide': 1}
CLASS_NAMES = ['non-suicide (0)', 'suicide (1)']
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

versions = {}
for package in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'gensim', 'xgboost']:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = 'not installed'
display(pd.Series(versions, name='Runtime version').to_frame())


## ۲. بارگذاری و کنترل کیفیت متن خام

پیش از تقسیم فقط کنترل‌های ثابت کیفیت انجام می‌شوند: ستون‌ها و برچسب‌ها،
متن تهی، تکرار متن خام و برچسب متناقض. هیچ واژگان، embedding یا آماری برای
تبدیل ویژگی‌ها از کل داده یاد گرفته نمی‌شود. کلید تکرار متن خام فقط
Unicode، بزرگی حروف و فاصله را یکسان می‌کند. ردیف‌های دارای برچسب متناقض
برای یک کلید خام حذف می‌شوند، سپس از هر متن تکراری یک نمونه باقی می‌ماند.

In [ ]:
def locate_dataset(explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'DATA_PATH does not exist: {path}')
        return path
    candidates = [Path('Suicide_Detection.csv'), Path('data/Suicide_Detection.csv')]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        candidates += sorted(kaggle_input.rglob('Suicide_Detection.csv'))
    existing = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
    if len(existing) > 1:
        raise ValueError('Several matching CSV files exist. Set DATA_PATH explicitly.')
    if existing:
        return existing[0]
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError('Set DATA_PATH to a local CSV, or install kagglehub.') from exc
    try:
        root = Path(kagglehub.dataset_download('nikhileswarkomati/suicide-watch'))
    except Exception as exc:
        raise RuntimeError('Download failed. Attach the CSV and set DATA_PATH; '
                           'Kaggle with Internet disabled needs an attached dataset.') from exc
    matches = sorted(root.rglob('Suicide_Detection.csv'))
    if len(matches) != 1:
        raise FileNotFoundError('Expected exactly one Suicide_Detection.csv in the download.')
    return matches[0]


def raw_duplicate_key(text):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', text).casefold()).strip()


def validate_raw_data(frame):
    missing_columns = {'text', 'class'} - set(frame.columns)
    if missing_columns:
        raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
    # Ignore CSV index columns instead of requiring a particular Unnamed: 0 column.
    data = frame.loc[:, ['text', 'class']].copy()
    data.insert(0, 'row_id', np.arange(len(data)))
    audit = {'loaded_rows': len(data)}
    invalid_text = ~data['text'].map(lambda value: isinstance(value, str) and bool(value.strip()))
    audit['invalid_or_empty_text_rows'] = int(invalid_text.sum())
    data = data.loc[~invalid_text].copy()
    audit['missing_label_rows'] = int(data['class'].isna().sum())
    data = data.loc[data['class'].notna()].copy()
    normalized_labels = data['class'].astype(str).str.strip().str.lower()
    unknown = sorted(set(normalized_labels) - set(LABEL_MAP))
    if unknown:
        raise ValueError(f'Unexpected labels: {unknown}; expected {list(LABEL_MAP)}')
    data['label'] = normalized_labels.map(LABEL_MAP).astype('int64')
    data['raw_key'] = data['text'].map(raw_duplicate_key)
    labels_per_key = data.groupby('raw_key', sort=False)['label'].nunique()
    conflicting_keys = labels_per_key.index[labels_per_key > 1]
    conflicting = data['raw_key'].isin(conflicting_keys)
    audit['raw_conflicting_label_rows'] = int(conflicting.sum())
    data = data.loc[~conflicting].copy()
    audit['raw_duplicate_rows'] = int(data.duplicated('raw_key').sum())
    data = data.drop_duplicates('raw_key', keep='first').reset_index(drop=True)
    audit['remaining_rows'] = len(data)
    if set(data['label']) != {0, 1} or data['label'].value_counts().min() < 10:
        raise ValueError('Both classes need at least 10 distinct usable texts for the split and CV.')
    return data, pd.Series(audit, name='Rows')


csv_path = locate_dataset(DATA_PATH)
raw_df = pd.read_csv(csv_path, usecols=['text', 'class'])
dataset, raw_audit = validate_raw_data(raw_df)
print(f'Dataset: {csv_path.name}')
display(raw_audit.to_frame())


## ۳. ابتدا تقسیم، سپس پاکسازی و پردازش

`train_test_split` روی متن خام و با `stratify` انجام می‌شود. Word2Vec هنوز
ساخته نشده است. اعتبارسنجی برای انتخاب مدل و آستانه است؛ تست برای گزارش نهایی.

In [ ]:
development_df, test_df = train_test_split(
    dataset, test_size=TEST_SIZE, random_state=SEED, stratify=dataset['label'])
train_df, validation_df = train_test_split(
    development_df, test_size=VALIDATION_FRACTION_OF_DEVELOPMENT,
    random_state=SEED, stratify=development_df['label'])
train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()

def assert_disjoint(partitions, column):
    seen = set()
    for frame in partitions:
        current = set(frame[column])
        assert not (seen & current), f'Overlap detected in {column}'
        seen.update(current)

assert_disjoint([train_df, validation_df, test_df], 'row_id')
assert_disjoint([train_df, validation_df, test_df], 'raw_key')
print('Raw split sizes:', {name: len(frame) for name, frame in
      [('train', train_df), ('validation', validation_df), ('test', test_df)]})


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFKC', html.unescape(text)).lower()
    text = text.replace('’', "'").replace('‘', "'")
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b', ' ', text)
    text = re.sub(r'(?<!\w)(?:u/|r/|@)[\w-]+', ' ', text)
    # Expand common English negations before punctuation removal.
    text = re.sub(r"\bcan't\b", 'can not', text)
    text = re.sub(r"\bwon't\b", 'will not', text)
    text = re.sub(r"\bshan't\b", 'shall not', text)
    text = re.sub(r"n['’]t\b", ' not', text)
    # Keep hashtag words, numbers and Unicode letters; no learned stopword list.
    text = re.sub(r'[^\w\s]', ' ', text)
    text = text.replace('_', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def clean_partitions(partitions):
    # Fixed priority: train -> validation -> test.
    # Held-out texts can never cause a training row to be removed or changed.
    # Collisions created by cleaning are removed from the later partition.
    result, audit, seen = {}, [], set()
    for name, source in partitions.items():
        part = source.copy()
        part['cleaned_text'] = part['text'].map(clean_text)
        record = {'split': name, 'before': len(part)}
        empty = part['cleaned_text'].eq('')
        record['empty_after_cleaning'] = int(empty.sum())
        part = part.loc[~empty].copy()
        overlap = part['cleaned_text'].isin(seen)
        record['duplicates_of_earlier_split'] = int(overlap.sum())
        part = part.loc[~overlap].copy()
        # Inspect labels only within this partition, never across held-out boundaries.
        counts = part.groupby('cleaned_text', sort=False)['label'].nunique()
        conflicting = part['cleaned_text'].isin(counts.index[counts > 1])
        record['local_conflicting_label_rows'] = int(conflicting.sum())
        part = part.loc[~conflicting].copy()
        record['local_duplicates'] = int(part.duplicated('cleaned_text').sum())
        part = part.drop_duplicates('cleaned_text').reset_index(drop=True)
        if set(part['label']) != {0, 1}:
            raise ValueError(f'{name} lost a class after cleaning; inspect the quality audit.')
        seen.update(part['cleaned_text'])
        record['after'] = len(part)
        result[name] = part
        audit.append(record)
    return result, pd.DataFrame(audit).set_index('split')


partitions, cleaning_audit = clean_partitions(
    {'train': train_df, 'validation': validation_df, 'test': test_df})
train_df, validation_df, test_df = (partitions[name] for name in ['train', 'validation', 'test'])
assert_disjoint([train_df, validation_df, test_df], 'cleaned_text')
assert_disjoint([train_df, validation_df, test_df], 'row_id')
if train_df['label'].value_counts().min() < CV_FOLDS:
    raise ValueError('Not enough training examples per class for CV_FOLDS.')
display(cleaning_audit)
display(pd.DataFrame({name: frame['label'].value_counts().reindex([0, 1], fill_value=0)
                      for name, frame in partitions.items()}).rename(index=dict(enumerate(CLASS_NAMES))))

# Pipelines receive raw strings; their tokenizer applies the same fixed cleaner.
X_train = train_df['text'].reset_index(drop=True)
X_validation = validation_df['text'].reset_index(drop=True)
X_test = test_df['text'].reset_index(drop=True)
y_train = train_df['label'].to_numpy()
y_validation = validation_df['label'].to_numpy()
y_test = test_df['label'].to_numpy()


## ۴. تحلیل اکتشافی فقط روی آموزش

برچسب نمودار به کد کلاس متصل است و به ترتیب تصادفی `value_counts` وابسته نیست.
توکن‌ها مستقیماً با `split` و `join` استفاده می‌شوند؛ تبدیل فهرست پایتون
به رشته که در نسخهٔ اصلی ویرگول وارد واژگان می‌کرد حذف شده است.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = train_df['label'].value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(CLASS_NAMES, counts.to_numpy(), color=['#26a6a6', '#465ac7'])
axes[0].set(title='Training class distribution', ylabel='Rows')
axes[1].hist(train_df['cleaned_text'].str.split().str.len(), bins=50, color='#465ac7')
axes[1].set(title='Training text length', xlabel='Tokens', ylabel='Rows')
fig.tight_layout()
plt.show()

positive_texts = train_df.loc[train_df['label'].eq(1), 'cleaned_text']
word_counts = Counter(token for text in positive_texts for token in text.split())
top_words = word_counts.most_common(20)
if top_words:
    words, frequencies = zip(*top_words)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(list(words)[::-1], list(frequencies)[::-1], color='#465ac7')
    ax.set(title='Most frequent tokens: training positive class', xlabel='Count')
    fig.tight_layout()
    plt.show()


In [ ]:
# Optional word cloud; model training does not depend on this package.
try:
    from wordcloud import WordCloud
except ImportError:
    print('wordcloud is optional; the frequency plot above is available.')
else:
    if word_counts:
        cloud = WordCloud(width=1000, height=400, background_color='white',
                          random_state=SEED).generate_from_frequencies(word_counts)
        plt.figure(figsize=(12, 5))
        plt.imshow(cloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word cloud: training positive class only')
        plt.show()


## ۵. Word2Vec قابل استفاده در Pipeline

`fit` واژگان و embedding را یاد می‌گیرد؛ `transform` فقط میانگین بردار
واژه‌های شناخته‌شده را می‌سازد. متن کاملاً ناشناخته بردار صفر می‌گیرد و
نرخ آن در پایان گزارش می‌شود. برای تکرارپذیری، یک worker و hash پایدار
به‌کار رفته‌اند؛ نسخهٔ کتابخانه‌ها نیز بالاتر ثبت می‌شود.

Cache فقط خروجی برازش همان ورودی و همان پارامترها را بازاستفاده می‌کند.
هر فولد ورودی متفاوت دارد و embedding جداگانه می‌گیرد. زمان‌های برازش
به‌علت بازاستفاده از cache، مقایسهٔ منصفانهٔ سرعت کامل الگوریتم‌ها نیستند.

In [ ]:
def stable_hash(value):
    return int.from_bytes(md5(str(value).encode('utf-8')).digest()[:4], 'little')


class TextTokenizer(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        return [clean_text(text).split() for text in X]


class MeanWord2Vec(TransformerMixin, BaseEstimator):
    def __init__(self, vector_size=100, window=5, min_count=2, sg=1,
                 epochs=5, workers=1, seed=42):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.epochs = epochs
        self.workers = workers
        self.seed = seed

    def fit(self, X, y=None):
        sentences = [list(tokens) for tokens in X]
        self.model_ = Word2Vec(
            vector_size=self.vector_size, window=self.window, min_count=self.min_count,
            sg=self.sg, epochs=self.epochs, workers=self.workers, seed=self.seed,
            hashfxn=stable_hash)
        self.model_.build_vocab(sentences)
        if not self.model_.wv.key_to_index:
            raise ValueError('Word2Vec vocabulary is empty. Check cleaning and min_count.')
        self.model_.train(sentences, total_examples=self.model_.corpus_count,
                          epochs=self.epochs)
        self.n_features_out_ = self.vector_size
        return self

    def transform(self, X):
        check_is_fitted(self, 'model_')
        rows = list(X)
        features = np.zeros((len(rows), self.vector_size), dtype=np.float32)
        for i, tokens in enumerate(rows):
            indices = [self.model_.wv.key_to_index[token] for token in tokens
                       if token in self.model_.wv.key_to_index]
            if indices:
                features[i] = self.model_.wv.vectors[indices].mean(axis=0)
        if not np.isfinite(features).all():
            raise ValueError('Non-finite values in document embeddings.')
        return features


cache_owner = tempfile.TemporaryDirectory(prefix='w2v_pipeline_') if USE_PIPELINE_CACHE else None
pipeline_memory = Memory(cache_owner.name, verbose=0) if cache_owner else None

def make_text_pipeline(classifier):
    return Pipeline([
        ('tokens', TextTokenizer()),
        ('w2v', MeanWord2Vec(**W2V_CONFIG)),
        ('classifier', classifier),
    ], memory=pipeline_memory)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)


## ۶. آموزش و جست‌وجوی پارامتر فقط با بخش آموزش

Random Forest و رأی‌گیری Naive Bayes تنظیمات ثابت دارند. برای Decision Tree و
XGBoost از `best_estimator_` جست‌وجو استفاده می‌شود؛ پارامترهای دستی جایگزین
نتیجهٔ جست‌وجو نمی‌شوند. XGBoost مسئلهٔ دودویی با `logloss` دارد، نه معیار رتبه‌بندی `map`.

embeddingهای Word2Vec می‌توانند منفی باشند؛ `MultinomialNB` به این ویژگی‌ها
نمی‌خورد و مدل استفاده‌نشدهٔ آن حذف شده است. رأی‌گیری نسخهٔ اصلی با GaussianNB
و BernoulliNB حفظ شده؛ `binarize=0.0` صریح است. StandardScaler در پایپ‌لاین
همین مدل فقط روی آموزش برازش می‌شود. این یک baseline روی embedding است و
لزوماً مزیت NB روی شمارش کلمات را ندارد.

In [ ]:
model_specs = {
    'Random Forest': (
        RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=MODEL_N_JOBS),
        None),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=SEED),
        {
            'classifier__criterion': ['gini', 'entropy', 'log_loss'],
            'classifier__max_depth': [3, 5, 8, 10, 15, 20, None],
            'classifier__min_samples_split': [2, 5, 10, 20, 50],
            'classifier__min_samples_leaf': [1, 5, 10, 20, 50],
            'classifier__max_features': [None, 'sqrt', 'log2'],
            'classifier__class_weight': [None, 'balanced'],
        }),
}
if RUN_XGBOOST:
    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError('Install xgboost and restart, or set RUN_XGBOOST=False.') from exc
    model_specs['XGBoost'] = (
        XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                      tree_method='hist', random_state=SEED, n_jobs=MODEL_N_JOBS),
        {
            'classifier__max_depth': [3, 5, 7, 10],
            'classifier__n_estimators': [50, 100, 150],
            'classifier__learning_rate': [0.05, 0.1, 0.2],
        })

nb_classifier = Pipeline([
    ('scale', StandardScaler()),
    ('vote', VotingClassifier(
        estimators=[('gaussian', GaussianNB(var_smoothing=1e-8)),
                    ('bernoulli', BernoulliNB(alpha=1.0, binarize=0.0))],
        voting='soft', n_jobs=1)),
])
model_specs['Naive Bayes voting'] = (nb_classifier, None)


In [ ]:
fitted_models, search_results, training_records = {}, {}, []
for name, (classifier, parameters) in model_specs.items():
    print(f'Fitting {name}...')
    pipeline = make_text_pipeline(classifier)
    started = time.perf_counter()
    if parameters is None:
        fitted_models[name] = pipeline.fit(X_train, y_train)
        best_cv_score = np.nan
    else:
        search = RandomizedSearchCV(
            estimator=pipeline, param_distributions=parameters,
            n_iter=SEARCH_ITERATIONS, scoring=SEARCH_SCORING, cv=cv,
            refit=True, random_state=SEED, n_jobs=SEARCH_N_JOBS,
            pre_dispatch=SEARCH_N_JOBS, error_score='raise', verbose=1,
            return_train_score=False)
        search.fit(X_train, y_train)
        fitted_models[name] = search.best_estimator_
        search_results[name] = search
        best_cv_score = search.best_score_
        print('Best parameters:', search.best_params_)
        print(f'Best training CV {SEARCH_SCORING}: {best_cv_score:.4f}')
    training_records.append({'model': name, 'best_cv_score': best_cv_score,
                             'fit_or_search_seconds': time.perf_counter() - started})
training_summary = pd.DataFrame(training_records).set_index('model')
display(training_summary)


## ۷. احتمال کلاس مثبت، معیارها و انتخاب آستانه روی اعتبارسنجی

هر مدل احتمال کلاس ۱ خودش را می‌دهد؛ AUC و Average Precision مستقیماً از
همین احتمال محاسبه می‌شوند، نه از برچسب آستانه‌خورده یا خروجی مدل قبلی.
معیارهای positive، macro و weighted نام مستقل دارند. Recall کلاس مثبت
جداگانه گزارش می‌شود؛ weighted recall در این مسئله با accuracy برابر است.
قرارداد تصمیم در همه‌جا `probability >= threshold` است.

In [ ]:
def positive_probability(estimator, X):
    classes = np.asarray(estimator.classes_)
    locations = np.flatnonzero(classes == 1)
    if len(locations) != 1:
        raise ValueError(f'Positive class 1 not found in {classes}')
    probabilities = estimator.predict_proba(X)[:, int(locations[0])]
    if not np.isfinite(probabilities).all():
        raise ValueError('Non-finite probabilities.')
    return probabilities


def metric_summary(y_true, probabilities, threshold):
    y_pred = (np.asarray(probabilities) >= threshold).astype('int64')
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_positive': precision_score(y_true, y_pred, zero_division=0),
        'recall_positive': recall_score(y_true, y_pred, zero_division=0),
        'f1_positive': f1_score(y_true, y_pred, zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'roc_auc': roc_auc_score(y_true, probabilities),
        'average_precision': average_precision_score(y_true, probabilities),
    }


def select_threshold(y_true, probabilities):
    rows = []
    for threshold in THRESHOLDS:
        predictions = (probabilities >= threshold).astype('int64')
        rows.append({
            'threshold': float(threshold),
            'accuracy': accuracy_score(y_true, predictions),
            'precision_positive': precision_score(y_true, predictions, zero_division=0),
            'recall_positive': recall_score(y_true, predictions, zero_division=0),
            'f1_positive': f1_score(y_true, predictions, zero_division=0),
        })
    curve = pd.DataFrame(rows)
    # Ties: higher precision, then closer to 0.5, then smaller threshold.
    ranked = curve.assign(distance_to_half=(curve['threshold'] - 0.5).abs()).sort_values(
        ['f1_positive', 'precision_positive', 'distance_to_half', 'threshold'],
        ascending=[False, False, True, True], kind='mergesort')
    return float(ranked.iloc[0]['threshold']), curve


In [ ]:
validation_rows, validation_probabilities, threshold_curves, selected_thresholds = [], {}, {}, {}
for name, estimator in fitted_models.items():
    probabilities = positive_probability(estimator, X_validation)
    validation_probabilities[name] = probabilities
    threshold, curve = select_threshold(y_validation, probabilities)
    threshold_curves[name] = curve
    selected_thresholds[name] = threshold
    validation_rows.append({'model': name, 'threshold': threshold,
                            **metric_summary(y_validation, probabilities, threshold)})

validation_table = pd.DataFrame(validation_rows).sort_values(
    ['f1_positive', 'f1_macro', 'roc_auc', 'model'],
    ascending=[False, False, False, True], kind='mergesort').reset_index(drop=True)
print('VALIDATION results — used for model/threshold selection; not final test estimates.')
display(validation_table)
best_name = validation_table.iloc[0]['model']
best_threshold = selected_thresholds[best_name]
best_pipeline = fitted_models[best_name]
print(f'Frozen choice: {best_name}, validation-selected threshold={best_threshold:.2f}')


In [ ]:
fig, axes = plt.subplots(len(threshold_curves), 1,
                         figsize=(10, 3.5 * len(threshold_curves)), squeeze=False)
for ax, (name, curve) in zip(axes.ravel(), threshold_curves.items()):
    for metric in ['accuracy', 'precision_positive', 'recall_positive', 'f1_positive']:
        ax.plot(curve['threshold'], curve[metric], label=metric)
    ax.axvline(selected_thresholds[name], linestyle='--', color='black', label='Selected')
    ax.set(title=f'{name}: validation only', xlabel='Threshold', ylabel='Metric', ylim=(0, 1.02))
    ax.legend(loc='best', fontsize=8)
fig.tight_layout()
plt.show()


## ۸. ارزیابی نهایی مدل منتخب روی تست

انتخاب مدل و آستانه در سلول قبل تمام شده است. مدل با validation مجدداً
آموزش داده نمی‌شود، چون توزیع احتمال و آستانه ممکن است تغییر کند.
تست فقط در این مرحله برای پیش‌بینی نهایی استفاده می‌شود. هر دو آستانهٔ
از پیش مشخص‌شدهٔ ۰٫۵ و منتخب اعتبارسنجی گزارش می‌شوند؛ از نتایج تست
برای تغییر انتخاب استفاده نکنید. مدل‌های دیگر فقط جدول اعتبارسنجی دارند.

In [ ]:
test_probabilities = positive_probability(best_pipeline, X_test)
test_rows = [
    {'setting': 'Default 0.50', 'threshold': 0.5,
     **metric_summary(y_test, test_probabilities, 0.5)},
    {'setting': 'Validation-selected', 'threshold': best_threshold,
     **metric_summary(y_test, test_probabilities, best_threshold)},
]
final_test_results = pd.DataFrame(test_rows).set_index('setting')
print(f'FINAL TEST — model: {best_name}')
display(final_test_results)
final_predictions = (test_probabilities >= best_threshold).astype('int64')
print(classification_report(y_test, final_predictions, labels=[0, 1],
                            target_names=CLASS_NAMES, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
cm = confusion_matrix(y_test, final_predictions, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap='Blues', values_format='d', colorbar=False)
axes[0].set_title(f'{best_name}: test, threshold={best_threshold:.2f}')
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('Final test ROC')
PrecisionRecallDisplay.from_predictions(y_test, test_probabilities, ax=axes[2], name=best_name)
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Positive prevalence')
axes[2].set_title('Final test precision-recall')
axes[2].legend()
fig.tight_layout()
plt.show()


## ۹. کنترل واژگان و پوشش embedding

این کنترل به واژگان مدل منتخب نگاه می‌کند؛ واژه‌ای که فقط در بخش‌های
نگه‌داشته‌شده دیده شده، نباید وارد واژگان آموزش شده باشد. وجود بردار صفر
برای متن کاملاً OOV خطای اجرا نیست؛ نرخ آن باید در تفسیر نتیجه دیده شود.

In [ ]:
training_tokens = {token for text in train_df['cleaned_text'] for token in text.split()}
heldout_tokens = {token for frame in [validation_df, test_df]
                  for text in frame['cleaned_text'] for token in text.split()}
heldout_only_tokens = heldout_tokens - training_tokens
learned_vocabulary = set(best_pipeline.named_steps['w2v'].model_.wv.key_to_index)
assert learned_vocabulary <= training_tokens, 'Vocabulary includes non-training tokens.'
assert learned_vocabulary.isdisjoint(heldout_only_tokens), 'Held-out vocabulary leakage.'

def embedding_coverage(texts, vocabulary):
    total_tokens = known_tokens = no_known_token_rows = 0
    for text in texts:
        tokens = clean_text(text).split()
        known = sum(token in vocabulary for token in tokens)
        total_tokens += len(tokens)
        known_tokens += known
        no_known_token_rows += int(known == 0)
    return {'rows': len(texts),
            'token_coverage': known_tokens / total_tokens if total_tokens else 0.0,
            'all_oov_rows': no_known_token_rows,
            'all_oov_fraction': no_known_token_rows / len(texts) if len(texts) else 0.0}

coverage_table = pd.DataFrame({
    name: embedding_coverage(texts, learned_vocabulary)
    for name, texts in [('train', X_train), ('validation', X_validation), ('test', X_test)]
}).T
display(coverage_table)
print('Vocabulary audit passed. Held-out-only tokens in vocabulary: 0')


## ۱۰. تغییرات و حدود ارزیابی

| ایراد نسخهٔ اصلی | اصلاح |
|---|---|
| آموزش Word2Vec پیش از تقسیم | تقسیم متن خام قبل از برازش |
| embedding مشترک میان آموزش و ارزیابی CV | Word2Vec داخل Pipeline هر فولد |
| بررسی آستانه با تست | تنظیم آستانه فقط با validation |
| استفادهٔ مکرر از تست برای مقایسه | انتخاب با validation و تست نهایی مدل منتخب |
| AUC درخت و XGBoost با احتمال Random Forest؛ AUC قبلی در NB | احتمال کلاس مثبت همان مدل |
| نادیده‌گرفتن پارامترهای برتر جست‌وجو | استفاده از best_estimator_ |
| نمایش امتیاز recall به‌عنوان Accuracy | نام‌گذاری صریح معیار؛ پیش‌فرض جست‌وجو F1 مثبت |
| رشته‌سازی فهرست توکن‌ها و ورود ویرگول به کلمات | انتقال مستقیم توکن‌ها |
| حذف نفی، return تکراری و اصلاح فاصلهٔ معیوب | پاکسازی سادهٔ ثابت با حفظ نفی |
| تکرار، متن تهی و برچسب نامعتبر | کنترل کیفیت و گزارش تعداد حذف‌ها |
| نمودار با برچسب کلاس وابسته به ترتیب شمارش | ترتیب ثابت ۰ و ۱ |
| تصادفی‌بودن مدل‌ها و Word2Vec چند worker | seed ثابت، یک worker و hash پایدار |

اعداد قبلی قابل مقایسهٔ مستقیم با اجرای جدید نیستند: تقسیم، کنترل کیفیت،
پاکسازی و انتخاب پارامتر اصلاح شده‌اند. معیار validation برای انتخاب
استفاده شده و برآورد بی‌طرف نهایی نیست؛ معیار test را جداگانه گزارش کنید.
این کنترل‌ها تکرار خام و تکرار پس از پاکسازی را پوشش می‌دهند؛ مشابهت معنایی،
بازنشرهای نزدیک و هم‌پوشانی نویسندگان را تضمین نمی‌کنند. اگر شناسهٔ نویسنده
یا زمان معتبر در دسترس باشد، تقسیم گروهی/زمانی متناسب با هدف پژوهش لازم است.
تستی که قبلاً بارها برای تصمیم‌گیری دیده شده با اصلاح کد «دیده‌نشده» نمی‌شود؛
ارزیابی مستقل آینده به دادهٔ نگه‌داشته‌شدهٔ تازه نیاز دارد.

راهنماهای اصلی:
[scikit-learn: data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)،
[Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline)،
[threshold tuning](https://scikit-learn.org/stable/modules/classification_threshold.html)،
[Gensim Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html).

In [ ]:
# Release temporary feature caches. Fitted pipelines remain usable for prediction.
# Re-running training cells recreates cached entries as needed.
if pipeline_memory is not None:
    pipeline_memory.clear(warn=False)
if cache_owner is not None:
    cache_owner.cleanup()
print('Run complete. Use final_test_results for the held-out test report.')
